# Step 21.4 — Top-k：限制候选集合后再采样

## 先建立独立工作副本（以下只是命令参考，本次未执行）

在仓库根目录先同步母版：

```bash
git pull
```

再创建不覆盖已有文件的时间戳副本（仓库应已有 notebooks/work 目录）：

```bash
cp -n notebooks/95_step21_4_top_k_sampling.ipynb notebooks/work/95_step21_4_top_k_sampling_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

只在工作副本中填写练习或未来运行。`cp -n` 在同名目标存在时不覆盖；同一秒再次操作也不会覆盖已有内容。当前母版是**材料初稿 0/2**，不是学习完成证明。

**执行边界：**所有代码单元 execution_count 为 null、outputs 为空。本次不执行 notebook、MLX 实验、测试、训练、验证、采样、语料读取/下载或 checkpoint 操作。下面的短实验是将来可观察的安排，不是已观察结果。MLX API 基线固定 v0.32.2。

## 1. 为什么需要这一节

完整词表里可能有大量很低分候选。Top-k 先保留最高分的 k 个候选，再在这个集合内抽样。它改变支持集（support），不是只让分布变尖。Temperature 改变候选之间的比例；top-k 直接使部分 token 不再可能出现。

本节明确采用“严格 k 个索引”的定义。另一种常见阈值法保留所有大于等于第 k 大分数的项，会因并列值留下超过 k 个候选。两种策略都能合理，但不能混写定义和实现。

## 2. 先备数学与 Python / MLX 语义

设 S 为保留索引集合，z'ᵢ=zᵢ（i∈S），否则为 −∞。因为 exp(−∞)=0，softmax(z') 的分母只包含 S：p'ᵢ=exp(zᵢ)/Σⱼ∈S exp(zⱼ)。categorical 接受这些 masked logits，内部按其定义采样，不需要先显式 softmax。

为清晰定义并列规则，本节用 Python 排序：先按分数降序，同分按 token ID 升序，再取前 k 个。这需要把小数组传到 host，适合四词玩具，不是高吞吐实现。此取舍比用不确定 tie 行为的排序假装严格可复现更透明。

## 3. 纸笔走一遍（讲解例子，不是练习答案）

z=[ln6,ln3,ln1]，未过滤时概率 [.6,.3,.1]。k=2 保留 ID 0,1 后概率变为 [2/3,1/3,0]，不是 [.6,.3,0]，因为必须重新归一化。k=1 时只剩一个候选；k=V 时支持集不变。

并列例子 z=[4,2,2,0]，k=2：本节约定保留 ID 0 和 1；若用 z≥2 的阈值则会保留 0,1,2，恰好显示两种算法的差别。

## 4. Shape tracing 与接口契约

单条 logits (V,) → host list 长 V → 排序得到 k 个原始 token ID → Boolean mask (V,) → masked logits (V,) → categorical 标量 ID。排序索引不是新的词表；输出必须仍指向原词表。接入批量 (B,V) 时需要逐行定义，本节不冒充已支持 batch。

In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def exact_top_k_logits(logits, k):
    if logits.ndim != 1 or logits.shape[0] == 0:
        raise ValueError("expect one nonempty vocabulary row (V,)")
    V = logits.shape[0]
    if type(k) is not int or not 1 <= k <= V:
        raise ValueError("k must be an integer in [1,V]")
    z = logits.astype(mx.float32)
    mx.eval(z)
    values = z.tolist()
    if any(not math.isfinite(x) for x in values):
        raise ValueError("input logits must be finite before filtering")
    selected = sorted(range(V), key=lambda i: (-values[i], i))[:k]
    keep = mx.array([i in selected for i in range(V)], dtype=mx.bool_)
    return mx.where(keep, z, float("-inf")), selected

## 5. 短实验计划

先预测 shape 与趋势，再在未来工作副本中运行下一个单元。这里没有运行记录。若未来结果和推导不同，先检查输入、axis、dtype、mask、随机 key，再讨论数值误差；不要把一次随机输出当成分布证明。

In [ ]:
z = mx.array([math.log(6), math.log(3), 0.], dtype=mx.float32)
filtered, selected = exact_top_k_logits(z, 2)
p = mx.softmax(filtered, axis=-1)
token = mx.random.categorical(filtered, key=mx.random.key(23))
mx.eval(p, token)
print("kept original IDs:", selected, "p:", p.tolist(), "draw:", token.item())

## 6. 常见误区与未来衔接

正温度缩放不改变排序，所以在精确算术下 top-k 集合不受温度影响；数值误差、并列规则仍需说明。若组合两个助手，先对有限原始 logits 做 temperature scaling，再交给本节 top-k，不能把 -Inf 喂给 21.3 的有限输入检查。

k 必须至少 1，否则所有位置被屏蔽，分布不存在。把被排除分数写成 0 不是屏蔽，因为 exp(0)=1。Top-k 不是 top-p/nucleus sampling；后者根据累计概率确定可变大小集合，不在本节实现。

## 7. 练习（不附答案）

1. 对四个不同分数手算 k=2 的归一化概率。
2. 设计包含三个并列最大值的例子，对比严格 k 与阈值法。
3. 解释为何返回排序后位置而不是原 token ID 会导致错误解码。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 8. 官方 API 阅读与完成边界

- [MLX v0.32.2 / python/src/random.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/random.cpp)
- [MLX v0.32.2 / python/src/ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)

接口说明以固定版本源码为准。本次只有文本与源代码静态检查，没有运行通过声明。能够复述概念与完成练习后再进入下一节；材料覆盖范围不代表读者已经掌握。